# Voice2Reel — نوتبوك واحد لمولّد الـ Stickman: بيانات ← تدريب ← فيديو

**هذا هو النوتبوك الوحيد الذي تشغّله لمسار الـ Stickman.** لا تحتاج تفتح `05` ولا `01` لهذا المسار.

يشغّل بالترتيب:

1. توليد/فحص **60,000 خطة رسم** مع `--check` و`scripts/check_data.py`.
2. تدريب بوابة الجودة على CPU: منحنيات `0.7615 → 0.8698 → 0.9304` واختبار `0.9299`.
3. تدريب **مولّد عصبي صغير (MLP)** على نقاط الوصل الـ 18، مع قياس خطأ المفاصل ومقارنته بنموذج ridge.
4. توليد **فيديو MP4** من نقاط المولّد عبر الراسم المتجهي الحتمي، مع ترجمة عربية وصوت اختياري.
5. كتابة تقرير `stickman_pipeline_report.json`، عرض العيّنات والفيديو، وتنزيل النتائج.

**حدود صريحة:** البيانات اصطناعية CC0 بدون مراجعة خبراء؛ المولّد يتنبأ بنقاط متجهية، **ليس** مولّد صور diffusion؛ الفيديو عرض توضيحي، **ليس** جودة الإنتاج؛ هذا النوتبوك **لا يدرّب Qwen** ولا يطلع `final_report.json` الخاص بكاتب السكربت. ذاك يبقى في `notebooks/05_quality_loop.ipynb` على T4 GPU. النموذج هنا صغير ويشتغل على CPU، وإن توفر T4 يستخدمه تلقائياً.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO = Path('/content/Voice2Reel')
if not (REPO / 'pyproject.toml').is_file():
    REPO = Path.cwd()
    while REPO != REPO.parent and not (REPO / 'pyproject.toml').is_file():
        REPO = REPO.parent
assert (REPO / 'pyproject.toml').is_file(), 'شغّل الدفتر من جذر المستودع أو ارفع ZIP المستودع'
sys.path.insert(0, str(REPO))

WORK = REPO / 'outputs' / 'stickman_pipeline'
FRAMES_DIR = WORK / 'frames'
SAMPLES_DIR = WORK / 'samples'
VIDEO_PATH = WORK / 'stickman_demo.mp4'
REPORT_PATH = WORK / 'stickman_pipeline_report.json'
MODEL_PATH = WORK / 'stickman_mlp.pt'
for folder in (WORK, FRAMES_DIR, SAMPLES_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Outputs (gitignored):', WORK)

## 0) تثبيت الحزم

Colab غالباً فيه `torch`؛ إن ما كان موجوداً يثبّته الدفتر. `imageio-ffmpeg` يجلب ffmpeg ثابتاً لتجميع MP4. `edge-tts` اختياري للصوت العربي؛ الافتراضي **صامت** لأن الخدمة عبر الإنترنت وليست مُرخّصة تجارياً في هذا المستودع.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
     'Pillow==13.0.0', 'arabic-reshaper==3.1.0',
     'imageio-ffmpeg==0.6.0', 'edge-tts==7.3.1'],
    cwd=REPO, check=True,
)
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], cwd=REPO, check=True)
    import torch
import numpy as np
import imageio_ffmpeg
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('ffmpeg', imageio_ffmpeg.get_ffmpeg_exe())

## 1) تجهيز البيانات والتحقق منها

`--check` يعيد بناء الملفات ويقارن البايتات. `check_data.py` يفحص كل تقسيمات v2 + stickman، عدم التسريب، الـ 40 فحص سلامة، ويكتب/يطابق تقرير القياس المحفوظ.

In [ ]:
build = subprocess.run(
    [sys.executable, 'scripts/build_stickman_drawing_data.py', '--check'],
    cwd=REPO, check=True, capture_output=True, text=True,
)
print(build.stdout)
check = subprocess.run(
    [sys.executable, 'scripts/check_data.py'],
    cwd=REPO, check=True, capture_output=True, text=True,
)
print(check.stdout)

## 2) تدريب بوابة الجودة على CPU وقياسها

نُدرّب نموذج ridge للتنبؤ بنقاط الوصل، وكاشف حدودي للأخطاء، ونعيد حساب الدورات الثلاث الموثّقة. هذه هي الأرقام المحفوظة أصلاً في `examples/stickman_drawing/artist_rounds.json`.

In [ ]:
from voice2reel.stickman_artist import (
    DefectDetector, StickmanArtist, draw_frame, is_professional,
    render_samples, rows, run_rounds, vocabulary,
)

CPU_REPORT = run_rounds(max_rounds=3)
(WORK / 'artist_rounds.json').write_text(
    json.dumps(CPU_REPORT, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
DETECTOR = DefectDetector().fit(rows('train'))
DETECTOR_ACCURACY = DETECTOR.accuracy(rows('test'))
SAMPLE_METADATA = render_samples(SAMPLES_DIR)
print('CPU validation rounds:', [item['score'] for item in CPU_REPORT['rounds']])
print('CPU final test:', CPU_REPORT['final_test_on_best_round']['score'])
print('Defect detector test accuracy:', DETECTOR_ACCURACY)
print('Integrity checks:', len(CPU_REPORT['integrity']), 'all true:', all(CPU_REPORT['integrity'].values()))
print('Rendered sample frames:', len(SAMPLE_METADATA))

## 3) تدريب «مولّد الـ Stickman» العصبي

نستخدم `torch` لتدريب MLP صغير: مدخله نية المشهد (إيماءة/كاميرا/مشاعر/طور الحركة/اللوحة…) ومخرجه **18 نقطة وصل × 2** بصيغة مُقيّسة إلى أبعاد 540×960. ندرّب على صفوف `train` السليمة فقط، ونقيس على `test` السليم. إن توفر `cuda` استُخدم، وإلا CPU كافٍ.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DESIGNER = StickmanArtist()
CANVAS = np.array([540.0, 960.0], dtype=np.float32)
TRAIN_PROF = [row for row in rows('train') if is_professional(row)]
TEST_PROF = [row for row in rows('test') if is_professional(row)]


def design_target(part):
    x = np.vstack([DESIGNER.design(row) for row in part]).astype(np.float32)
    y = np.vstack([
        np.asarray(row['points'], dtype=np.float32).reshape(-1, 2) / CANVAS
        for row in part
    ]).reshape(len(part), -1).astype(np.float32)
    return x, y

X_TRAIN, Y_TRAIN = design_target(TRAIN_PROF)
MEAN = X_TRAIN.mean(axis=0)
STD = np.where(X_TRAIN.std(axis=0) < 1e-6, 1.0, X_TRAIN.std(axis=0)).astype(np.float32)
X_TRAIN = ((X_TRAIN - MEAN) / STD).astype(np.float32)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(20261110)
np.random.seed(20261110)
MODEL = nn.Sequential(
    nn.Linear(X_TRAIN.shape[1], 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, Y_TRAIN.shape[1]),
).to(DEVICE)
OPTIMIZER = torch.optim.Adam(MODEL.parameters(), lr=1e-3)
LOSS_FN = nn.MSELoss()
LOADER = DataLoader(
    TensorDataset(torch.from_numpy(X_TRAIN), torch.from_numpy(Y_TRAIN)),
    batch_size=256, shuffle=True,
)
EPOCHS = 30
HISTORY = []
for epoch in range(1, EPOCHS + 1):
    MODEL.train()
    total = 0.0
    for xb, yb in LOADER:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        OPTIMIZER.zero_grad()
        loss = LOSS_FN(MODEL(xb), yb)
        loss.backward()
        OPTIMIZER.step()
        total += float(loss.item()) * len(xb)
    HISTORY.append(round(total / len(LOADER.dataset), 6))
print('device:', DEVICE, '| train rows:', len(TRAIN_PROF), '| input:', X_TRAIN.shape[1], '| output:', Y_TRAIN.shape[1])
print('loss:', HISTORY[0], '->', HISTORY[-1])

In [ ]:
def predict_points(row):
    x = ((DESIGNER.design(row) - MEAN) / STD).astype(np.float32)
    with torch.no_grad():
        MODEL.eval()
        output = MODEL(torch.from_numpy(x).unsqueeze(0).to(DEVICE)).cpu().numpy()
    points = output.reshape(18, 2) * CANVAS
    points = np.clip(points, [0.0, 0.0], CANVAS)
    return [[round(float(a), 2), round(float(b), 2)] for a, b in points]


def mean_joint_error(part, predictor):
    errors = []
    for row in part:
        predicted = np.asarray(predictor(row), dtype=np.float64).reshape(-1, 2)
        reference = np.asarray(row['points'], dtype=np.float64).reshape(-1, 2)
        errors.append(float(np.mean(np.linalg.norm(predicted - reference, axis=1))))
    return round(float(np.mean(errors)), 3)

EVAL_ROWS = TEST_PROF[:1000]
MLP_ERROR = mean_joint_error(EVAL_ROWS, predict_points)
RIDGE = StickmanArtist().fit(TRAIN_PROF)
RIDGE_ERROR = mean_joint_error(EVAL_ROWS, RIDGE.predict)
torch.save({
    'state_dict': MODEL.state_dict(),
    'mean': MEAN.tolist(), 'std': STD.tolist(),
    'input_dim': int(X_TRAIN.shape[1]), 'output_dim': int(Y_TRAIN.shape[1]),
    'canvas': CANVAS.tolist(), 'epochs': EPOCHS,
}, MODEL_PATH)
print(json.dumps({
    'mlp_mean_joint_error_px': MLP_ERROR,
    'ridge_mean_joint_error_px': RIDGE_ERROR,
    'defect_accuracy': DETECTOR_ACCURACY,
    'model_saved': str(MODEL_PATH),
}, ensure_ascii=False, indent=2))

## 4) توليد الفيديو من نقاط المولّد

ثمانية مشاهد × 2.5 ثانية × 12 إطار/ثانية = 240 إطاراً. كل إطار يأخذ طور حركة مختلف، يتنبأ المولّد بنقاطه، ثم يرسمه `draw_frame` مع عنوان عربي ولوحة معلومات وترجمة سفلية. المخرجات في `outputs/` (مستثناة من Git).

In [ ]:
VOCAB = vocabulary()
JOINT_GAP_INDEX = VOCAB['feature_names'].index('joint_gap')
BASE_BY_CAMERA = {
    camera: next(row for row in rows('validation')
                 if is_professional(row) and row['camera'] == camera)
    for camera in ('wide', 'medium', 'close')
}
SCENES = [
    ('ابدأ بمثال', 'present', 'wide', 'none', 'neutral', 'hook'),
    ('قسّم المسألة', 'point', 'wide', 'list', 'happy', 'explain'),
    ('افحص المدخل', 'think', 'medium', 'code', 'concerned', 'explain'),
    ('قارن الحالتين', 'shrug', 'medium', 'compare', 'thinking', 'explain'),
    ('صحح الترتيب', 'thumbs_up', 'medium', 'flow', 'happy', 'explain'),
    ('اختبر جزءا صغيرا', 'present', 'close', 'code', 'neutral', 'explain'),
    ('راجع النتيجة', 'wave', 'wide', 'list', 'surprised', 'close'),
    ('خلاصة المشهد', 'present', 'wide', 'none', 'happy', 'close'),
]
FPS = 12
SECONDS_PER_SCENE = 2.5
FRAMES_PER_SCENE = int(FPS * SECONDS_PER_SCENE)
previous_gesture = 'wave'
frame_number = 0
bounds_ok = True
for scene_index, (caption, gesture, camera, visual, emotion, intent) in enumerate(SCENES):
    base = BASE_BY_CAMERA[camera]
    for frame_index in range(FRAMES_PER_SCENE):
        phase = round(frame_index / max(1, FRAMES_PER_SCENE - 1), 3)
        frame_row = dict(base)
        frame_row.update({
            'id': f'generated_{scene_index:02d}_{frame_index:04d}',
            'split': 'generated', 'defect': None,
            'caption': caption, 'gesture': gesture,
            'previous_gesture': previous_gesture,
            'camera': camera, 'emotion': emotion,
            'intent': intent, 'visual': visual,
            'panel_visible': visual != 'none',
            'phase': phase,
            'features': list(base['features']),
            'style': dict(base['style']),
            'actor_box': list(base['actor_box']),
        })
        frame_row['features'][JOINT_GAP_INDEX] = 0.0
        predicted = predict_points(frame_row)
        bounds_ok &= all(0.0 <= x <= 540.0 and 0.0 <= y <= 960.0 for x, y in predicted)
        draw_frame(frame_row, points=predicted,
                   output=FRAMES_DIR / f'frame_{frame_number:05d}.png')
        frame_number += 1
    previous_gesture = gesture
print('frames:', frame_number, '| all predicted joints inside canvas:', bool(bounds_ok))
print('scenes:', [scene[0] for scene in SCENES])

## 5) صوت اختياري + تجميع MP4

الافتراضي `USE_EDGE_TTS = False` فيطلع فيديو بمسار صوت صامت. غيّره إلى `True` إن أردت تجربة صوت عربي عبر `edge-tts` (خدمة خارجية، غير مُرخّصة تجارياً هنا). عند أي فشل يرجع الدفتر تلقائياً للصامت.

In [ ]:
USE_EDGE_TTS = False
NARRATION = ' . '.join(scene[0] for scene in SCENES) + ' .'
AUDIO_PATH = WORK / 'narration.mp3'
AUDIO_MODE = 'silent'
if USE_EDGE_TTS:
    try:
        import asyncio
        import edge_tts

        async def synthesize():
            communicator = edge_tts.Communicate(NARRATION, 'ar-SA-ZariyahNeural')
            await communicator.save(str(AUDIO_PATH))

        asyncio.run(synthesize())
        AUDIO_MODE = 'edge-tts ar-SA-ZariyahNeural'
    except Exception as exc:
        print('Edge TTS skipped, using silent audio:', repr(exc))
        AUDIO_MODE = 'silent'

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
command = [
    FFMPEG, '-y', '-loglevel', 'error',
    '-framerate', str(FPS), '-i', str(FRAMES_DIR / 'frame_%05d.png'),
]
if AUDIO_MODE != 'silent' and AUDIO_PATH.is_file():
    command += ['-i', str(AUDIO_PATH)]
else:
    command += ['-f', 'lavfi', '-i', 'anullsrc=channel_layout=stereo:sample_rate=44100']
command += [
    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-r', str(FPS),
    '-c:a', 'aac', '-shortest', '-movflags', '+faststart', str(VIDEO_PATH),
]
subprocess.run(command, check=True)
print('video:', VIDEO_PATH, '| bytes:', VIDEO_PATH.stat().st_size, '| audio:', AUDIO_MODE)
print('duration seconds:', round(frame_number / FPS, 2))

## 6) التقرير، العرض والتنزيل

`stickman_pipeline_report.json` يجمع كل شيء: البيانات، التدريب، القياس، الفيديو، والحدود. هذا **ليس** `final_report.json` الخاص بكاتب `Qwen` في النوتبوك 05.

In [ ]:
from IPython.display import Image, Video, display

PIPELINE_REPORT = {
    'notebook': '06_stickman_generator.ipynb',
    'data': {
        'total_rows': 60000,
        'splits': {'train': 50000, 'validation': 5000, 'test': 5000},
        'keypoints': len(VOCAB['keypoint_order']),
        'features': len(VOCAB['feature_names']),
        'defects': len(VOCAB['defects']),
        'provenance': 'assistant-authored synthetic CC0-1.0 vector plans; no expert review',
    },
    'training': {
        'device': str(DEVICE), 'epochs': EPOCHS,
        'train_professional_rows': len(TRAIN_PROF),
        'final_mse_loss': HISTORY[-1],
        'mlp_mean_joint_error_px': MLP_ERROR,
        'ridge_mean_joint_error_px': RIDGE_ERROR,
        'defect_detector_test_accuracy': DETECTOR_ACCURACY,
        'model_weights': str(MODEL_PATH),
    },
    'cpu_quality_loop': {
        'validation_scores': [item['score'] for item in CPU_REPORT['rounds']],
        'final_test_score': CPU_REPORT['final_test_on_best_round']['score'],
        'integrity_checks': len(CPU_REPORT['integrity']),
        'integrity_all_true': all(CPU_REPORT['integrity'].values()),
    },
    'video': {
        'path': str(VIDEO_PATH), 'frames': frame_number, 'fps': FPS,
        'duration_seconds': round(frame_number / FPS, 2),
        'scenes': [scene[0] for scene in SCENES],
        'audio': AUDIO_MODE,
        'predicted_joints_inside_canvas': bool(bounds_ok),
        'renderer': 'deterministic vector draw_frame; not diffusion/raster artwork',
    },
    'limitations': [
        'No Qwen/writer training and no Qwen final_report.json here; use notebook 05 on T4 GPU.',
        'The MLP predicts vector joints only; it is not a neural image generator.',
        'The demo video is not production video/voice quality and has no human expert review.',
        'Silent audio is default; edge-tts is optional and not cleared for commercial use.',
    ],
}
REPORT_PATH.write_text(
    json.dumps(PIPELINE_REPORT, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(PIPELINE_REPORT, ensure_ascii=False, indent=2))

for name in ('sample_00_present.png', 'sample_04_bad_proportions.png', 'sample_07_overlap.png'):
    print(name)
    display(Image(filename=str(SAMPLES_DIR / name)))
display(Video(str(VIDEO_PATH), embed=True))

try:
    from google.colab import files
    for artifact in (VIDEO_PATH, REPORT_PATH, MODEL_PATH):
        files.download(str(artifact))
    print('Downloaded video, report and model weights.')
except ImportError:
    print('Not running in Colab; artifacts are in', WORK)

## بعد التشغيل — بصراحة

- هذا النوتبوك أنجز مسار **الـ Stickman** كاملاً في دفتر واحد: بيانات ← تدريب ← قياس ← فيديو ← تقرير.
- **ما أنجز مسار «كلهم» كاملاً**: كاتب السكربت على T4، `final_report.json`، مراجعة بشرية للنصوص، تسجيل أصوات بترخيص، وتقييم صوت/فيديو حقيقي كلها ما زالت تتطلب `notebooks/05_quality_loop.ipynb` وعمل لاحق.
- لا يوجد ادّعاء 90% عن المنتج، ولا ادّعاء جمال مهني؛ `0.9299` قياس CPU على بيانات اصطناعية فقط.
- لو أردت مولّد صور عصبي حقيقي لاحقاً، نحتاج corpus صور raster بتراخيص موثّقة وتدريب GPU منفصل.